# AIDE JupyterLab 本地部署（单图/批量推理）

这个 Notebook 用于在 JupyterLab 中部署 AIDE，并输出你关注的变量：`P(fake)`（AI 生成概率）。


In [ ]:
import os
import json
from pathlib import Path

import torch
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from torchvision import transforms

from data.dct import DCT_base_Rec_Module
from models.AIDE import AIDE

print('torch:', torch.__version__)

## 1) 配置路径
- `checkpoint_path`：你训练好的 AIDE checkpoint（必须）
- `resnet_pretrained_path` / `convnext_pretrained_path`：可选。
  - 当 checkpoint 已包含完整权重时，可保持 `None`。
  - 如果你遇到权重缺失报错，再填这两个路径。


In [ ]:
checkpoint_path = 'pretrained_ckpts/checkpoint-best.pth'  # 必填
resnet_pretrained_path = None  # e.g. 'pretrained_ckpts/resnet50.pth'
convnext_pretrained_path = None  # e.g. 'pretrained_ckpts/open_clip_pytorch_model.bin'
device = 'cuda' if torch.cuda.is_available() else 'cpu'

print('device =', device)
print('checkpoint exists =', Path(checkpoint_path).exists())

## 2) 加载模型

In [ ]:
class AIDEPredictor:
    def __init__(self, checkpoint_path, device='cuda', resnet_path=None, convnext_path=None):
        if device == 'cuda' and not torch.cuda.is_available():
            device = 'cpu'
        self.device = torch.device(device)

        self.model = AIDE(resnet_path=resnet_path, convnext_path=convnext_path).to(self.device)
        checkpoint = torch.load(checkpoint_path, map_location='cpu')
        state_dict = checkpoint.get('model', checkpoint)

        missing, unexpected = self.model.load_state_dict(state_dict, strict=False)
        if missing:
            print('Missing keys:', len(missing))
        if unexpected:
            print('Unexpected keys:', len(unexpected))

        self.model.eval()

        self.to_tensor = transforms.ToTensor()
        self.resize = transforms.Resize([256, 256])
        self.normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        self.dct = DCT_base_Rec_Module()

    @torch.no_grad()
    def predict(self, image_or_path):
        if isinstance(image_or_path, (str, Path)):
            image = Image.open(image_or_path).convert('RGB')
            image_path = str(image_or_path)
        else:
            image = image_or_path.convert('RGB')
            image_path = '<PIL.Image>'

        image_tensor = self.to_tensor(image)
        x_minmin, x_maxmax, x_minmin1, x_maxmax1 = self.dct(image_tensor)

        x_0 = self.normalize(self.resize(image_tensor))
        x_minmin = self.normalize(self.resize(x_minmin))
        x_maxmax = self.normalize(self.resize(x_maxmax))
        x_minmin1 = self.normalize(self.resize(x_minmin1))
        x_maxmax1 = self.normalize(self.resize(x_maxmax1))

        x = torch.stack([x_minmin, x_maxmax, x_minmin1, x_maxmax1, x_0], dim=0).unsqueeze(0)
        x = x.to(self.device, non_blocking=True)

        logits = self.model(x)
        probs = torch.softmax(logits, dim=-1)[0]
        p_real = float(probs[0].item())
        p_fake = float(probs[1].item())

        return {
            'image_path': image_path,
            'p_real': p_real,
            'p_fake': p_fake,
            'label': 'fake' if p_fake >= 0.5 else 'real'
        }

predictor = AIDEPredictor(
    checkpoint_path=checkpoint_path,
    device=device,
    resnet_path=resnet_pretrained_path,
    convnext_path=convnext_pretrained_path
)
print('Model loaded.')

## 3) 单图推理
将 `image_path` 改成你自己的图片路径。

In [ ]:
image_path = 'demo.jpg'

result = predictor.predict(image_path)
print(json.dumps(result, indent=2, ensure_ascii=False))

img = Image.open(image_path).convert('RGB')
plt.figure(figsize=(5, 5))
plt.imshow(img)
plt.axis('off')
plt.title(f"P(fake)={result['p_fake']:.4f} | Pred={result['label']}")
plt.show()

## 4) 批量推理并导出 CSV（可选）

In [ ]:
from glob import glob

image_dir = 'images'
output_csv = 'aide_predictions.csv'
exts = ('*.jpg', '*.jpeg', '*.png', '*.webp', '*.bmp')

paths = []
for ext in exts:
    paths.extend(glob(str(Path(image_dir) / ext)))

rows = []
for p in paths:
    try:
        rows.append(predictor.predict(p))
    except Exception as e:
        rows.append({'image_path': p, 'p_real': None, 'p_fake': None, 'label': f'error: {e}'})

df = pd.DataFrame(rows)
df.to_csv(output_csv, index=False)
print(f'saved: {output_csv}, total: {len(df)}')
df.head()